# LTFormer Colab 复现指南（极速多进程 + 断点续传版）

> **LTFormer**: *A light-weight transformer-based self-supervised matching network for heterogeneous remote sensing images* (*Information Fusion*, 2024)

### 使用流程建议
1. **阶段一（CPU High-RAM 运行时）**：运行 **步骤 1 ~ 步骤 4**，利用多核 CPU + Colab 本地 SSD 极速将 `whu-opt-sar` 原始大图切分并生成自监督三元组，按分片（`chunk_XXXX.tar`）实时保存到 Google Drive。**如中途断开，重连后依次重跑步骤 1~4 即可自动跳过已完成的分片断点续传。**
2. **阶段二（GPU 运行时）**：三元组全部生成完成后，在 Colab 菜单选择 `代码执行程序 -> 更改运行时类型 -> GPU`，依次运行 **步骤 1、步骤 2、步骤 5、步骤 6**，将分片快速解压到本地 SSD 并开始训练，权重实时保存回 Google Drive。

### 步骤 1: 挂载 Google Drive 并检查当前硬件环境

In [ ]:
from google.colab import drive
import os, psutil, torch

drive.mount('/content/drive')

print(f"CPU 核心数: {os.cpu_count()}")
print(f"系统总内存: {psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"GPU 可用: {torch.cuda.is_available()}", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

### 步骤 2: 克隆 / 更新 Fork 仓库并安装运行依赖

In [ ]:
import os
%cd /content
if not os.path.exists('/content/article-LTFormer'):
    !git clone https://github.com/jabin-hao/article-LTFormer.git
else:
    %cd /content/article-LTFormer
    !git pull

%cd /content/article-LTFormer
!pip install -q hydra-core omegaconf timm tifffile pyyaml tqdm
!ls -la

### 步骤 3: 配置 Google Drive 工作区路径并探查原始大图规格

In [ ]:
import os, glob
from run_generate_dataset_fast import collect_images, read_image_any

WORKSPACE_DIR = "/content/drive/MyDrive/paper/LTFormer_workspace"
RAW_OPTICAL_DIR = os.path.join(WORKSPACE_DIR, "data/whu-opt-sar/optical")
RAW_SAR_DIR = os.path.join(WORKSPACE_DIR, "data/whu-opt-sar/sar")
DRIVE_CHUNKS_DIR = os.path.join(WORKSPACE_DIR, "triplets_chunks")
DRIVE_TRAIN_CKPT_DIR = os.path.join(WORKSPACE_DIR, "train_outputs")

os.makedirs(DRIVE_CHUNKS_DIR, exist_ok=True)
os.makedirs(DRIVE_TRAIN_CKPT_DIR, exist_ok=True)

opt_files = collect_images(RAW_OPTICAL_DIR)
sar_files = collect_images(RAW_SAR_DIR)
print(f"optical 图像数: {len(opt_files)}")
print(f"sar     图像数: {len(sar_files)}")

if opt_files:
    sample_opt = read_image_any(opt_files[0])
    print(f"示例 optical 文件: {os.path.basename(opt_files[0])}, shape={sample_opt.shape}, dtype={sample_opt.dtype}")
if sar_files:
    sample_sar = read_image_any(sar_files[0])
    print(f"示例 sar     文件: {os.path.basename(sar_files[0])}, shape={sample_sar.shape}, dtype={sample_sar.dtype}")

### 步骤 4（CPU 运行时）: 多核极速 + 断点续传生成三元组
- `--mode auto`：若 `optical` 为 4 波段（RGB+NIR），按论文 README 自动剥离可见光与近红外波段并切分成 `926×926`；若为 3 波段，则自动与 `sar` 配对并切分成 `926×926`。
- 源图先并发缓存到 Colab 本地 SSD `/content/ltformer_cache`，处理完每个分片后立即打包为 `chunk_XXXX.tar` 保存到 Google Drive。
- 若中途断连，重新运行本单元格会自动跳过 Drive 中已生成的 `chunk_*.tar`。

In [ ]:
import os, glob

!python run_generate_dataset_fast.py \
    --rgb-dir "{RAW_OPTICAL_DIR}" \
    --pair-dir "{RAW_SAR_DIR}" \
    --out-dir "{DRIVE_CHUNKS_DIR}" \
    --cache-dir "/content/ltformer_cache" \
    --mode auto \
    --tile-size 926 \
    --patch-half 128 \
    --chunk-size 5

chunks = sorted(glob.glob(os.path.join(DRIVE_CHUNKS_DIR, "chunk_*.tar")))
total_mb = sum(os.path.getsize(c) for c in chunks) / (1024 ** 2)
print(f"\n✅ Google Drive 当前共有 {len(chunks)} 个三元组分片，总计 {total_mb:.1f} MB")

### 步骤 5（切换 GPU 运行时后执行）: 从 Google Drive 解压三元组到本地 SSD
切到 GPU 运行时后先执行 **步骤 1 ~ 步骤 3**，再执行本单元格。把分片解压到 `/content/triplets_train`，避免训练时直接读写 Drive 导致 GPU 饿死。

In [ ]:
import os, glob, tarfile
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "⚠️ 当前未检测到 GPU！请先切换 Colab 运行时为 GPU 后再继续。"

LOCAL_TRAIN_DIR = "/content/triplets_train"
os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)

chunk_tars = sorted(glob.glob(os.path.join(DRIVE_CHUNKS_DIR, "chunk_*.tar")))
assert len(chunk_tars) > 0, f"未在 {DRIVE_CHUNKS_DIR} 找到 chunk_*.tar，请先完成步骤 4！"

def extract_one(tar_path):
    with tarfile.open(tar_path, "r") as tar:
        tar.extractall(LOCAL_TRAIN_DIR)

with ThreadPoolExecutor(max_workers=4) as ex:
    list(tqdm(ex.map(extract_one, chunk_tars), total=len(chunk_tars), desc="解压三元组到本地SSD"))

num_folders = len(os.listdir(LOCAL_TRAIN_DIR))
print(f"🎉 解压完成！本地可用于训练的三元组样本文件夹数: {num_folders}")

### 步骤 6（GPU 运行时）: 启动自监督训练（Checkpoint 自动保存至 Google Drive）

In [ ]:
!python run_training_ltformer.py \
    paths.train_data="/content/triplets_train" \
    hydra.run.dir="{DRIVE_TRAIN_CKPT_DIR}"

print(f"✅ 训练结束，模型权重已保存在: {DRIVE_TRAIN_CKPT_DIR}")